In [ ]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import os
import matplotlib.pyplot as plt
import tifffile as tiff
from torchvision import transforms
from PIL import Image
import numpy as np
import torch.nn.functional as F



In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
import torchvision.transforms as T

class TestDataset(Dataset):
    def __init__(self, img_dir, mask_dir):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        
        self.img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.tif')])
        self.mask_files = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith('.tif')])
        assert len(self.img_files) == len(self.mask_files)

        # ✅ Make inference-scale match RandomResizedCrop target, but without rotation expand
        self.preprocess = T.Compose([
            T.Resize((768, 768), interpolation=T.InterpolationMode.BILINEAR),
        ])
    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img = tiff.imread(os.path.join(self.img_dir, self.img_files[idx]))
        mask = tiff.imread(os.path.join(self.mask_dir, self.mask_files[idx]))

        if img.ndim == 3:
            img = img[img.shape[0]//2]
        if mask.ndim == 3:
            mask = mask[mask.shape[0]//2]

        img = img.astype(np.float32)
        img = img / img.max()  # tone matches training

        mask = (mask > 0).astype(np.float32)

        img = Image.fromarray(img)
        mask = Image.fromarray(mask)

        img = self.preprocess(img)
        mask = self.preprocess(mask)

        img = torch.tensor(np.array(img)).unsqueeze(0)
        mask = torch.tensor(np.array(mask)).unsqueeze(0)
        return img, mask


In [ ]:
# -----------------------
# Transform
# -----------------------
transform = transforms.Compose([
    transforms.ToTensor(),  # convert to [0,1] tensor
])

In [ ]:
# -----------------------
# Load dataset
# -----------------------
test_img_dir = "./datas/original/test/imgs"

test_mask_dir = "./datas/original/test/labels"

test_ds = TestDataset(test_img_dir, test_mask_dir)


In [ ]:
# -----------------------
# Load model
# -----------------------
from models.baseline_Unet_ViT import U_net_ViT  # <- adjust import

model = U_net_ViT(
    encode_in=(1, 64, 128, 256),
    encode_out=(64, 128, 256, 512),
    decode_in=(1024, 512, 256, 128),
    decode_out=(512, 256, 128, 64),
    normalize=True
) # create model instance
model.load_state_dict(torch.load("./models/predicted_models/unet_vit_trained.pth", map_location=device))
model.to(device)
model.eval()

In [ ]:
import torch.nn.functional as F
import numpy as np
import os
import matplotlib.pyplot as plt
import tifffile as tiff

# ---------- dice util (same as before) ----------
def dice_score(pred, gt, eps=1e-6):
    pred, gt = pred.reshape(-1), gt.reshape(-1)
    inter = (pred * gt).sum()
    union = pred.sum() + gt.sum()
    return (2 * inter + eps) / (union + eps)

# ---------- prediction loop ----------
all_dices = []
N = len(test_ds)

save_dir = "./models/predicted_models/predictions/test"
os.makedirs(save_dir, exist_ok=True)

with torch.no_grad():
    for i in range(N):
        img, gt = test_ds[i]           # [1,H,W]
        H0, W0 = img.shape[-2:]       # original spatial size (already 768 in your case)

        img_batch = img.unsqueeze(0).to(device)  # [1,1,H,W]
        pred_logits = model(img_batch)
        pred_prob = torch.sigmoid(pred_logits).cpu()

        # Resize prediction back to GT size (if network output smaller)
        pred_up = F.interpolate(pred_prob, size=(H0, W0), mode='bilinear', align_corners=False)

        # Threshold exactly like validation
        pred_mask = (pred_up > 0.5).float()

        # Compute dice for stats
        d = dice_score(pred_mask.squeeze(), gt.squeeze()).item()
        all_dices.append(d)

        # Save files (optional)
        tiff.imwrite(os.path.join(save_dir, f"pred_{i}.tif"), pred_mask.squeeze().numpy().astype(np.uint8))
        tiff.imwrite(os.path.join(save_dir, f"gt_{i}.tif"), gt.squeeze().numpy().astype(np.uint8))

        # ----------- 🔍 PLOTS YOU REQUESTED -----------
        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        axs[0].imshow(img.squeeze().numpy(), cmap='gray')
        axs[0].set_title("Input Image (Original)")
        axs[1].imshow(gt.squeeze().numpy(), cmap='gray')
        axs[1].set_title("Ground Truth")
        axs[2].imshow(pred_mask.squeeze().numpy(), cmap='gray')
        axs[2].set_title(f"Predicted Mask (Dice={d:.4f})")

        for ax in axs:
            ax.axis('off')
        plt.show()

        # ----------- histograms for debugging -----------
        plt.hist(img.squeeze().numpy().flatten(), bins=100)
        plt.title("Intensity Histogram (Input)")
        plt.show()

        plt.hist(pred_up.squeeze().numpy().flatten(), bins=50)
        plt.title("Probability Histogram (Prediction)")
        plt.show()

        print(f"[{i+1}/{N}] Dice: {d:.4f}")

# ----------- 📌 FINAL TEST DICE STATS -----------
mean_dice = np.mean(all_dices)
std_dice = np.std(all_dices)
print(f"\nTEST SET RESULTS")
print(f"Images: {N}")
print(f"Mean Dice: {mean_dice:.4f}")
print(f"Std Dice: {std_dice:.4f}")
